# 03 — Pipeline Comparison

Compare all 5 RAG strategies on the same queries:
1. Naive RAG
2. Hybrid RAG
3. Reranker RAG
4. GraphRAG
5. Agentic RAG (via Deep Agent)

Requires indexes to be built first (run `02_indexing_pipeline.ipynb` or `scripts/ingest.py`).

In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
# Load indexes (cached after first call)
from src.tools.retrieval_tools import _get_store, _get_bm25, _get_graph
store = _get_store()
bm25 = _get_bm25()
graph = _get_graph()
print("Indexes loaded.")

## Test Queries

In [ ]:
TEST_QUERIES = [
    {"q": "Luật Đất đai 2024 có hiệu lực từ ngày nào?", "type": "factual"},
    {"q": "Nghị định nào sửa đổi Nghị định 43/2014/NĐ-CP?", "type": "multi_hop"},
    {"q": "Quy định về bảo hiểm xã hội nào còn hiệu lực năm 2025?", "type": "temporal"},
    {"q": "Doanh nghiệp nhỏ và vừa cần nộp những loại thuế gì?", "type": "reasoning"},
]

## Strategy comparison

In [ ]:
from src.evaluation.benchmark import STRATEGIES
from src.tools.retrieval_tools import generate_answer_tool

def run_strategy(strategy, query, k=3):
    t0 = time.perf_counter()
    docs = STRATEGIES[strategy](query, k=k)
    latency_ms = (time.perf_counter() - t0) * 1000
    docs_json = json.dumps([{"page_content": d.page_content, "metadata": d.metadata} for d in docs])
    answer = generate_answer_tool.invoke({"query": query, "docs_json": docs_json})
    return answer, latency_ms, docs

for item in TEST_QUERIES[:2]:  # Run first 2 queries to save API cost
    print(f"\n{'='*70}")
    print(f"Query [{item['type']}]: {item['q']}")
    print('='*70)
    for strategy in ['naive', 'hybrid', 'reranker', 'graph', 'agentic']:
        answer, latency_ms, docs = run_strategy(strategy, item['q'])
        print(f"\n[{strategy.upper()}] ({latency_ms:.0f}ms, {len(docs)} docs retrieved)")
        print(answer[:300])